In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath("/home1/smaruj/ledidi_akita/"))
from utils.df_utils import load_parameter_results

In [2]:
_PROJ     = "/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita"
RESULTS_DIR = f"{_PROJ}/optimizations/boundaries"

FOLDS   = [0, 1, 2, 3]
LAMBDAS = [0.01, 0.1, 1.0, 10.0, 100.0, 125.0, 200.0]

In [3]:
# Lambda sweep
df_lambda = load_parameter_results(RESULTS_DIR, "lambda", LAMBDAS, folds=FOLDS)

Total windows loaded: 1218


In [4]:
df_lambda["no_edits"] = df_lambda["n_edits"] == 0

no_edits_summary = (
    df_lambda.groupby("lambda")["no_edits"]
    .agg(n_total="count", n_no_edits="sum")
    .assign(pct_no_edits=lambda x: 100 * x["n_no_edits"] / x["n_total"])
)
print(no_edits_summary.to_string())

        n_total  n_no_edits  pct_no_edits
lambda                                   
0.01        174           0      0.000000
0.10        174           0      0.000000
1.00        174           0      0.000000
10.00       174           1      0.574713
100.00      174          14      8.045977
125.00      174          18     10.344828
200.00      174          29     16.666667


In [5]:
df_lambda["insul_diff"] = df_lambda["insul_score_edited"] - df_lambda["insul_score_orig"]
df_lambda["no_improvement"] = df_lambda["insul_diff"] >= 0

no_improvement_summary = (
    df_lambda.groupby("lambda")["no_improvement"]
    .agg(n_total="count", n_no_improvement="sum")
    .assign(pct_no_improvement=lambda x: 100 * x["n_no_improvement"] / x["n_total"])
)
print(no_improvement_summary.to_string())

        n_total  n_no_improvement  pct_no_improvement
lambda                                               
0.01        174                 0            0.000000
0.10        174                 0            0.000000
1.00        174                 0            0.000000
10.00       174                 1            0.574713
100.00      174                14            8.045977
125.00      174                18           10.344828
200.00      174                29           16.666667


In [6]:
success_rate = (
    df_lambda.assign(success=~df_lambda["no_edits"] & ~df_lambda["no_improvement"])
    .groupby("lambda")["success"]
    .agg(n_total="count", n_success="sum")
    .assign(pct_success=lambda x: 100 * x["n_success"] / x["n_total"])
)
print(success_rate.to_string())

        n_total  n_success  pct_success
lambda                                 
0.01        174        174   100.000000
0.10        174        174   100.000000
1.00        174        174   100.000000
10.00       174        173    99.425287
100.00      174        160    91.954023
125.00      174        156    89.655172
200.00      174        145    83.333333


In [7]:
summary = (
    df_lambda.groupby("lambda")
    .agg(
        n_success        = ("n_edits",          "count"),
        mean_n_edits     = ("n_edits",          "mean"),
        mean_insul_diff  = ("insul_diff",        "mean"),
        mean_CTCFs       = ("CTCFs_num",         "mean"),
    )
    .round(3)
)
print(summary.to_string())

        n_success  mean_n_edits  mean_insul_diff  mean_CTCFs
lambda                                                      
0.01          174       792.098           -0.314       7.121
0.10          174       757.385           -0.315       7.063
1.00          174       430.977           -0.306       6.655
10.00         174       226.253           -0.302       6.092
100.00        174        63.006           -0.218       3.270
125.00        174        51.115           -0.202       2.839
200.00        174        31.943           -0.169       2.144
